# Enhanced Animal Shelter Dashboard
MongoDB performs the filtering, sorting, pagination, and breed aggregation used by this dashboard.

In [ ]:
from dash import Dash, Input, Output, dash_table, dcc, html
import dash_leaflet as dl
import plotly.express as px

from CRUD_Python_Module import AnimalShelter


db = AnimalShelter()

DISPLAY_COLUMNS = [
    "animal_id",
    "name",
    "animal_type",
    "breed",
    "sex_upon_outcome",
    "age_upon_outcome_in_weeks",
    "location_lat",
    "location_long",
]

app = Dash(__name__)

# Custom table actions let MongoDB handle paging and sorting.
app.layout = html.Div(
    [
        html.H1("Animal Shelter Dashboard - Shawn Bethune"),
        html.Hr(),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {
                    "label": "Mountain or Wilderness Rescue",
                    "value": "mountain",
                },
                {
                    "label": "Disaster or Individual Tracking",
                    "value": "disaster",
                },
                {"label": "Reset", "value": "reset"},
            ],
            value="reset",
            inline=True,
        ),
        html.Hr(),
        dash_table.DataTable(
            id="datatable-id",
            columns=[{"name": column, "id": column} for column in DISPLAY_COLUMNS],
            data=[],
            page_action="custom",
            page_current=0,
            page_size=10,
            page_count=0,
            sort_action="custom",
            sort_mode="single",
            sort_by=[{"column_id": "animal_id", "direction": "asc"}],
            row_selectable="single",
            selected_rows=[],
        ),
        html.Hr(),
        html.Div(
            className="row",
            style={"display": "flex", "gap": "20px"},
            children=[
                html.Div(id="graph-id", style={"width": "50%"}),
                html.Div(id="map-id", style={"width": "50%"}),
            ],
        ),
    ]
)


# A new filter invalidates the current page and selected row.
@app.callback(
    Output("datatable-id", "page_current"),
    Input("filter-type", "value"),
)
def reset_page_when_filter_changes(_filter_type):
    return 0


@app.callback(
    Output("datatable-id", "data"),
    Output("datatable-id", "page_count"),
    Output("datatable-id", "selected_rows"),
    Input("filter-type", "value"),
    Input("datatable-id", "page_current"),
    Input("datatable-id", "page_size"),
    Input("datatable-id", "sort_by"),
)
def update_table(filter_type, page_current, page_size, sort_by):
    sort_field = "animal_id"
    sort_order = "asc"

    if sort_by:
        sort_field = sort_by[0]["column_id"]
        sort_order = sort_by[0]["direction"]

    # Send the current table state to MongoDB and return only one page.
    result = db.search_animals(
        filter_type=filter_type,
        page=page_current,
        page_size=page_size,
        sort_field=sort_field,
        sort_order=sort_order,
    )
    return result["records"], result["total_pages"], []


@app.callback(
    Output("graph-id", "children"),
    Input("filter-type", "value"),
)
def update_graph(filter_type):
    # Breed totals come from a MongoDB aggregation, not the visible table page.
    counts = db.breed_counts(filter_type)
    if not counts:
        return html.P("No matching animals were found.")

    figure = px.pie(
        counts,
        names="breed",
        values="count",
        title="Breed Distribution",
    )
    return dcc.Graph(figure=figure)


@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "data"),
    Input("datatable-id", "selected_rows"),
)
def update_map(records, selected_rows):
    if not records:
        return html.P("Select a filter with matching animals to view the map.")

    # Reset a stale selection after filtering or paging changes the table rows.
    selected_index = selected_rows[0] if selected_rows else 0
    if selected_index >= len(records):
        selected_index = 0

    animal = records[selected_index]
    latitude = animal.get("location_lat")
    longitude = animal.get("location_long")
    if latitude is None or longitude is None:
        return html.P("The selected animal does not have location data.")

    return dl.Map(
        style={"width": "100%", "height": "500px"},
        center=[latitude, longitude],
        zoom=10,
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(animal.get("breed", "Unknown breed")),
                    dl.Popup(
                        [
                            html.H3("Animal Name"),
                            html.P(animal.get("name", "Unknown")),
                        ]
                    ),
                ],
            ),
        ],
    )


app.run(jupyter_mode="inline", debug=False)
